# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryam884/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*



## Unit of analysis + time window

One row represents the daily performance of one content item for one pseudonymized client on one report date.

I use the fact_content_daily_performance dataset.

The analysis uses March 2026 (month = '2026-03') because it is a mid-panel month and avoids using the final month for model development.

In [17]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT
COUNT(*) AS total_rows,
MIN(report_date) AS first_day,
MAX(report_date) AS last_day
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
""").df()

,total_rows,first_day,last_day
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields

### Features
- gsc_impressions
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users

### Label
Future content performance (for example future clicks or traffic depending on the modeling task).

### Context
- report_date
- client_hash_id
- content_hash_id
- month

### Excluded
- Future information because it would cause data leakage.
- June 2026 because it is the final evaluation month.

In [18]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
LIMIT 1
""").df()


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Query 1 (Grain)
Query 2 (Counts + Window)
Query 3 (Availability)

In [12]:
rel = "hf://datasets/FlyRank/internship-warehouse"


In [13]:
con.sql(f"""
SELECT
report_date,
client_hash_id,
content_hash_id
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72


In [14]:
con.sql(f"""
SELECT
COUNT(*) AS total_rows,
MIN(report_date) AS first_day,
MAX(report_date) AS last_day
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_day,last_day
0,9841378,2026-03-01,2026-03-31


In [15]:
con.sql(f"""
SELECT
COUNT(*) AS rows_with_gsc
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
AND gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_with_gsc
0,3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


## Data limits

This dataset cannot explain why users behaved a certain way.

It contains historical Search Console and GA4 measurements but does not capture external factors such as marketing campaigns or seasonality.

The history length differs across clients, making the dataset an unbalanced panel.

The final month should not be used for feature engineering because it can introduce data leakage.

In [19]:
con.sql(f"""
SELECT
COUNT(DISTINCT client_hash_id) AS clients,
COUNT(DISTINCT content_hash_id) AS content_items
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,clients,content_items
0,55,331437


In [8]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print(" Connected to Hugging Face")

 Connected to Hugging Face


In [9]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,count_star()
0,78835655


In [10]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
LIMIT 1
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [11]:
con.sql(f"""
SELECT *
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## Five Features

| Feature | Available at decision time because... |
|---------|----------------------------------------|
| gsc_impressions | They are historical impressions already recorded before making a prediction. |
| gsc_clicks | They are historical clicks known before prediction. |
| gsc_avg_position | The average search position is already measured in past data. |
| ga4_pageviews | Pageviews have already happened and are available before prediction. |
| ga4_sessions | Sessions are historical user activity available at prediction time. |

In [20]:
rel = "hf://datasets/FlyRank/internship-warehouse"

features = con.sql(f"""
SELECT
gsc_impressions,
gsc_clicks,
gsc_avg_position,
ga4_pageviews,
ga4_sessions
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
LIMIT 10
""").df()

features

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,20,0,3.350000,<NA>,<NA>
1,1,0,0.000000,<NA>,<NA>
2,125,1,4.928000,<NA>,<NA>
3,7,0,4.000000,<NA>,<NA>
4,11,0,2.272727,<NA>,<NA>
5,239,1,7.347280,<NA>,<NA>
6,191,0,7.832461,<NA>,<NA>
7,55,0,3.272727,<NA>,<NA>
8,77,0,5.636364,<NA>,<NA>
9,2,0,4.500000,<NA>,<NA>


## Leakage Demonstration

If a future value (for example, future clicks) is accidentally included as a feature, the model will appear unrealistically accurate because it already contains the answer.

After removing that feature, the model's performance becomes more realistic.

This demonstrates why label leakage must be avoided.

In [21]:
demo = con.sql(f"""
SELECT
gsc_impressions,
gsc_clicks,
ga4_pageviews,

-- This is an intentionally leaked column
gsc_clicks AS leaked_label

FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
LIMIT 10
""").df()

demo

,gsc_impressions,gsc_clicks,ga4_pageviews,leaked_label
0,20,0,<NA>,0
1,1,0,<NA>,0
2,125,1,<NA>,1
3,7,0,<NA>,0
4,11,0,<NA>,0
5,239,1,<NA>,1
6,191,0,<NA>,0
7,55,0,<NA>,0
8,77,0,<NA>,0
9,2,0,<NA>,0


In [22]:
honest = con.sql(f"""
SELECT
gsc_impressions,
gsc_clicks,
ga4_pageviews

FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
LIMIT 10
""").df()

honest

,gsc_impressions,gsc_clicks,ga4_pageviews
0,20,0,<NA>
1,1,0,<NA>
2,125,1,<NA>
3,7,0,<NA>
4,11,0,<NA>
5,239,1,<NA>
6,191,0,<NA>
7,55,0,<NA>
8,77,0,<NA>
9,2,0,<NA>


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.